# 08 — Explorer et tester les fonctions de graphiques

Ce notebook appelle **directement** les fonctions de `chart_utils` / `report_builder` et affiche
les figures **Plotly nativement** dans le notebook — **sans page HTML intermédiaire**. Objectif :
voir chaque fonction, son appel réel (mêmes découpages que `report_builder`) et la tester.

- Édite les **paramètres** (`ENTITE`, `GHU`, `APPAREIL`, `ORGANE`, `ANNEE`) dans la cellule de
  préambule, puis ré-exécute les sections voulues.
- Chaque cellule montre l'**appel exact** (avec la signature en commentaire) et rend la figure.
- Les fonctions de survie/délais se **filtrent elles-mêmes** (on leur passe la table + l'entité).
  Les autres reçoivent un petit **découpage** (`tot`, `ghu_slice`, `melt_sejours`, `reg_tot`)
  reproduisant ce que `report_builder` prépare avant l'appel.

> À lancer depuis `notebooks/` (chemins `../data`). Une combinaison sans données peut lever une
> exception — c'est normal en exploration.

In [1]:
import sys
from pathlib import Path
from IPython.display import display
sys.path.insert(0, "../src")

from report_builder import (load_aphp, load_regional, load_survival, load_delais_hopitaux,
                            _market_share_evolution)
from chart_utils import (
    line_evolution, bar_comparison, stacked_treatments, donut_market_share,
    heatmap_appareils, heatmap_organes, waterfall_trends, regional_comparison,
    bar_appareils_years, treemap_organes,
    survival_by_stage, survival_evolution, survival_hospital_comparison,
    delay_evolution, delay_comparison_bar, delay_hospital_comparison,
    GHU_LIST, TREATMENT_COLS, REGIONAL_COLORS)

DATA_DIR = Path("../data").resolve()
MODE = "fictif"        # "reel" ou "fictif" : uniquement pour les mappings des comparaisons hôpitaux

aphp = load_aphp(DATA_DIR); reg = load_regional(DATA_DIR)
surv = load_survival(DATA_DIR); delais_hop = load_delais_hopitaux(DATA_DIR)

appareils = sorted(aphp[aphp.appareil != "TOTAL"].appareil.unique())
organes_by_app = {a: sorted(aphp[(aphp.entite == "AP-HP") & (aphp.appareil == a)
                                 & (aphp.organe != "TOTAL")].organe.unique()) for a in appareils}

# ── Paramètres (édite puis ré-exécute) ──
ENTITE   = "AP-HP"           # "AP-HP" ou un GHU (cf. GHU_LIST)
GHU      = GHU_LIST[0]       # GHU pour la part de marché vs AP-HP
APPAREIL = None              # None → 1er appareil ; ou p.ex. "SEIN"
ORGANE   = None              # None → 1er organe de l'appareil ; ou p.ex. "Colon-Rectum-Anus"
ANNEE    = None              # None → dernière année

APPAREIL = APPAREIL if APPAREIL in appareils else appareils[0]
ORGANE   = ORGANE if ORGANE in organes_by_app[APPAREIL] else organes_by_app[APPAREIL][0]
ANNEE    = int(ANNEE) if ANNEE else int(aphp.annee.max())

# ── Découpages (ce que report_builder prépare avant d'appeler les graphes) ──
def tot(df, e):
    return df[(df.entite == e) & (df.appareil == "TOTAL") & (df.organe == "TOTAL")].sort_values("annee")
def ghu_slice(df, appareil="TOTAL", organe="TOTAL"):
    return df[df.entite.isin(GHU_LIST) & (df.appareil == appareil) & (df.organe == organe)]
def melt_sejours(s):
    m = s.melt(id_vars=["annee"], value_vars=list(TREATMENT_COLS.keys()),
               var_name="type_sejour", value_name="nb_sejours")
    m["label"] = m["type_sejour"].map(TREATMENT_COLS); return m
reg_tot = reg[(reg.appareil == "TOTAL") & (reg.organe == "TOTAL")]

# Mappings hôpital→GHU (comparaisons inter-hôpitaux) selon MODE
if MODE == "fictif":
    from generateur_fictif import HOPITAL2GHU
    MAP_SURV = MAP_DELAIS = HOPITAL2GHU
else:
    try:
        from chargeur_long import mapping_hopital_ghu, mapping_hopital_ghu_delais
        MAP_SURV   = mapping_hopital_ghu(str(DATA_DIR), fictif=False)
        MAP_DELAIS = mapping_hopital_ghu_delais(str(DATA_DIR), fictif=False)
    except Exception as e:
        from generateur_fictif import HOPITAL2GHU
        print("⚠ mapping réel indisponible → repli fictif :", type(e).__name__)
        MAP_SURV = MAP_DELAIS = HOPITAL2GHU

print(f"MODE={MODE} · ENTITE={ENTITE} · GHU={GHU} · APPAREIL={APPAREIL} · ORGANE={ORGANE} · ANNEE={ANNEE}")

MODE=fictif · ENTITE=AP-HP · GHU=GHU Centre · APPAREIL=APPAREIL DIGESTIF · ORGANE=Autre (appareil digestif) · ANNEE=2025


## A. Patients & évolution

In [2]:
# line_evolution(df, x, y, group, title ; y_label, entities, show_covid, y_zero)
sl = tot(aphp, ENTITE)
display(line_evolution(sl, "annee", "nb_patients", "entite",
                       f"Évolution du nombre de patients — {ENTITE}"))
display(line_evolution(sl, "annee", "nb_nouveaux_patients", "entite",
                       f"Évolution des nouveaux patients — {ENTITE}"))

# bar_comparison(df, x, y, group, title ; y_label, barmode, entities)
display(bar_comparison(ghu_slice(aphp), "annee", "nb_patients", "entite",
                       "Patients par GHU — évolution", barmode="group", entities=GHU_LIST))

# waterfall_trends(df_entity ; title)
display(waterfall_trends(sl, f"Variation annuelle du nombre de patients — {ENTITE}"))

# bar_appareils_years(df ; entity, years, value_col)
display(bar_appareils_years(aphp))

## B. Séjours (par mode de prise en charge)

In [3]:
# Séjours « fondus » : melt des 4 modes → colonne label (= ce que fait report_builder)
sej = melt_sejours(tot(aphp, ENTITE))

# line_evolution sur les séjours
display(line_evolution(sej, "annee", "nb_sejours", "label",
                       f"Séjours par mode de prise en charge — {ENTITE}",
                       entities=list(TREATMENT_COLS.values())))

# stacked_treatments(df, group_col, title ; orientation) — définie, non utilisée par les pages.
# ATTENTION : attend un tableau LARGE (colonnes nb_sejours_*) + un group_col (ici l'année),
# et NON le tableau fondu `sej` (sinon toutes les traces sont sautées → figure vide).
display(stacked_treatments(tot(aphp, ENTITE), "annee", f"Répartition des séjours — {ENTITE}"))

## C. Parts de marché & contexte régional

In [8]:
# donut_market_share(df_year, entity_col, value_col, title ; entities, color_map)
gl = ghu_slice(aphp); gl = gl[gl.annee == ANNEE]
display(donut_market_share(gl, "entite", "nb_patients", f"Répartition par GHU — {ANNEE}"))

rl = reg_tot[reg_tot.annee == ANNEE]
# entities = types d'établissement présents (sinon donut_market_share filtre sur GHU_LIST → VIDE)
display(donut_market_share(rl, "entite", "nb_patients",
                           f"Répartition par type d'établissement — {ANNEE}",
                           entities=sorted(rl["entite"].unique()), color_map=REGIONAL_COLORS))

# regional_comparison(df_reg, y_col, title ; highlight, color_map)
display(regional_comparison(reg_tot, "nb_patients",
                            "Patients — AP-HP vs contexte régional", color_map=REGIONAL_COLORS))

# _market_share_evolution(ent_df, aphp_df, entity, libelle) : part d'un GHU dans l'AP-HP
display(_market_share_evolution(tot(aphp, GHU), tot(aphp, "AP-HP"), GHU, "TOTAL"))

## D. Vues appareil / organe (treemap, heatmaps)

In [5]:
# treemap_organes(df, entity, appareil, year ; entity_col, value_col)
display(treemap_organes(aphp, ENTITE, APPAREIL, ANNEE))

# heatmap_appareils(df, entity ; value_col, title) — définie, non utilisée actuellement
display(heatmap_appareils(aphp, ENTITE))

# heatmap_organes(df, entity, appareil ; entity_col, value_col, title) — idem
display(heatmap_organes(aphp, ENTITE, APPAREIL))

## E. Survie

In [ ]:
# survival_by_stage(df_surv, entity, appareil ; organe, year, population)
display(survival_by_stage(surv, ENTITE, APPAREIL))   # année = dernière avec survie (repli interne)

# survival_evolution(df_surv, entity, appareil ; organe, stade, population)
display(survival_evolution(surv, ENTITE, APPAREIL, stade="I-III"))
display(survival_evolution(surv, ENTITE, APPAREIL, stade="IV"))

# survival_hospital_comparison(df_surv, mapping ; appareil, organe, stade, population, annee)
display(survival_hospital_comparison(surv, MAP_SURV, appareil="TOTAL", stade="I-III"))

## F. Délais de prise en charge

In [7]:
# delay_evolution(df, entity, appareil ; organe, entity_col)
display(delay_evolution(aphp, ENTITE, APPAREIL))

# delay_comparison_bar(df, appareil, year ; entity_col, entities)
ents = aphp[aphp.entite.isin(["AP-HP"] + list(GHU_LIST))]
display(delay_comparison_bar(ents, APPAREIL, ANNEE))

# delay_hospital_comparison(df_aphp, mapping ; appareil, organe, annee)
display(delay_hospital_comparison(delais_hop, MAP_DELAIS, appareil="TOTAL", organe="TOTAL", annee=ANNEE))